In [1]:
!pip install ipywidgets numpy -q

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import numpy as np
import random

# ============================================================
# GAME STATE
# ============================================================
game = {
    "level": 1,
    "probes_left": 5,
    "electron_pos": None,
    "wave_width": None,
    "score": 0,
    "total_attempts": 0,
    "message": ""
}

# ============================================================
# STYLING
# ============================================================
style = """
<style>
.ripple-box { padding: 20px; border-radius: 15px; background: linear-gradient(135deg, #0d1b2a, #1b2838); color: white; font-family: 'Courier New', monospace; }
.title { color: #4fc3f7; font-size: 26px; font-weight: bold; }
.wave { color: #64b5f6; }
.danger { color: #ff7043; }
.success { color: #66bb6a; }
.lesson { background: #0d1117; padding: 15px; border-radius: 8px; margin: 10px 0; border-left: 3px solid #4fc3f7; color: #ccc; font-size: 14px; }
.story { color: #aaa; font-size: 15px; font-style: italic; }
.highlight { color: #ffa726; font-weight: bold; }
</style>
"""
display(HTML(style))

game_output = widgets.Output()

# ============================================================
# WAVE FUNCTION
# ============================================================
def probability_at(x, center, width):
    return np.exp(-((x - center)**2) / (2 * width**2))

def show_wave_visual(center, width, highlight_x=None):
    """ASCII wave visualization"""
    visual = ""
    for i in range(11):
        prob = probability_at(i, center, width)
        bar_height = int(prob * 20)
        marker = "🔵" if highlight_x is not None and i == highlight_x else " "
        bar = "█" * bar_height
        label = f"Pos {i}: "
        visual += f"{marker}{label}{bar} (p={prob:.2f})\n"
    return visual

# ============================================================
# LEVEL SETUP
# ============================================================
def setup_level():
    game["electron_pos"] = random.uniform(1, 9)
    if game["level"] == 1:
        game["wave_width"] = 2.5  # Wide wave - easy
        game["probes_left"] = 5
    elif game["level"] == 2:
        game["wave_width"] = 1.5  # Narrower - harder
        game["probes_left"] = 4
    elif game["level"] == 3:
        game["wave_width"] = 0.8  # Very narrow - expert
        game["probes_left"] = 3

# ============================================================
# DISPLAY
# ============================================================
def show_game(message="", reveal=False):
    with game_output:
        clear_output(wait=True)

        html = f"""
        <div class='ripple-box'>
            <div class='title'>🌊 RIPPLE: The Electron Heist</div>
            <div class='story'>"The electron is not a marble. It's a wave. Catch it... if you can."</div>
            <hr style='border-color: #333;'>
            <div style='background: #0d1117; padding: 12px; border-radius: 8px; margin: 8px 0;'>
                🎯 <b>Level {game['level']}</b> |
                🔬 Probes left: <b class='highlight'>{game['probes_left']}</b> |
                ⭐ Score: <b>{game['score']}</b>
            </div>
            <div style='background: #0d1117; padding: 12px; border-radius: 8px; margin: 8px 0; font-family: monospace; font-size: 13px;'>
                <b>🌊 Electron Wave:</b><br>
                {show_wave_visual(game['electron_pos'], game['wave_width'])}
            </div>
            <div style='background: #0d1117; padding: 15px; border-radius: 8px; margin: 8px 0; min-height: 80px;'>
                {message}
            </div>
        </div>
        """
        display(HTML(html))

# ============================================================
# PLAYER ACTION
# ============================================================
def probe(guess):
    if game["probes_left"] <= 0:
        return

    game["probes_left"] -= 1
    game["total_attempts"] += 1

    center = game["electron_pos"]
    width = game["wave_width"]
    prob = probability_at(guess, center, width)
    distance = abs(guess - center)

    msg = f"<b>🔬 You probed position {guess}</b><br><br>"
    msg += f"Probability density at {guess}: <b>{prob:.3f}</b><br>"

    if distance < 0.3:
        # CAUGHT!
        game["score"] += 10
        msg += f"<br><h2 class='success'>⚡ ELECTRON CAUGHT!</h2>"
        msg += f"<p>True position was <b>{center:.2f}</b>. You were within 0.3!</p>"

        # Lesson
        msg += f"""<div class='lesson'>
            <b>🌊 QUANTUM LESSON:</b><br>
            You found the electron — but notice: it wasn't exactly at any point BEFORE you measured.<br>
            <b>The wave function collapsed AT your measurement.</b><br><br>
            In Bohr's model, the electron had a fixed orbit. Schrödinger showed it's a PROBABILITY CLOUD.<br>
            You don't find the electron. You COLLAPSE the wave. Where you look determines what you see.<br><br>
            <b>This is why Eve can't intercept quantum messages.</b> Looking = collapsing = leaving a trace.
        </div>"""

        # Next level
        if game["level"] < 3:
            game["level"] += 1
            setup_level()
            msg += f"<br><h3 class='success'>⬆️ Level Up! Welcome to Level {game['level']}!</h3>"
            msg += f"<p>The wave is narrower now. Fewer probes. Harder to catch.</p>"
        else:
            msg += f"<br><h2 class='success'>🏆 YOU MASTERED THE QUANTUM WAVE!</h2>"
            msg += f"<p>Final Score: <b>{game['score']}</b></p>"
            msg += f"""<div class='lesson'>
                <b>🎓 WHAT YOU LEARNED:</b><br>
                1. Electrons are waves, not marbles — described by Schrödinger's equation<br>
                2. The wave function gives PROBABILITY, not position<br>
                3. Measurement COLLAPSES the wave — you create the result by looking<br>
                4. This is the physics behind quantum security — you can't observe without disturbing<br>
                5. RAQT and BB84 both rely on this principle<br>
            </div>"""
    elif distance < 1.0:
        msg += f"<div class='wave'>🌊 <b>Warm!</b> The electron is near. Wave is stronger here.</div>"
    elif distance < 2.0:
        msg += f"<div style='color: #ffa726;'>🤔 <b>Lukewarm.</b> Some probability, but not the peak.</div>"
    else:
        msg += f"<div class='danger'>❄️ <b>Cold.</b> The wave is low here. Electron is elsewhere.</div>"

    # Out of probes
    if game["probes_left"] <= 0 and distance >= 0.3:
        msg += f"<br><div class='danger'><h3>💨 ELECTRON ESCAPED!</h3></div>"
        msg += f"<p>True position was <b>{center:.2f}</b>. You ran out of probes.</p>"
        msg += f"""<div class='lesson'>
            <b>🌊 QUANTUM LESSON:</b><br>
            Too many measurements disturb the system. Each probe narrowed the wave,<br>
            but the electron's momentum changed. The uncertainty principle protects it.<br><br>
            <b>In quantum cryptography:</b> Eve faces the same problem. Every interception<br>
            disturbs the qubit. Too many attempts = guaranteed detection.
        </div>"""
        # Reset level
        setup_level()
        msg += f"<p class='story'>Retrying Level {game['level']}...</p>"

    show_game(msg)

# ============================================================
# BUTTONS
# ============================================================
def create_buttons():
    buttons = []
    for i in range(11):
        btn = widgets.Button(
            description=f"Pos {i}",
            button_style='primary' if i % 3 == 0 else 'info',
            layout=widgets.Layout(width='65px', height='40px', margin='2px')
        )
        btn.on_click(lambda b, pos=i: probe(pos))
        buttons.append(btn)
    return buttons

btn_reset = widgets.Button(
    description='🔄 New Game',
    button_style='danger',
    layout=widgets.Layout(width='120px', height='40px')
)

btn_lesson = widgets.Button(
    description='📚 What is the Wave?',
    button_style='warning',
    layout=widgets.Layout(width='180px', height='40px')
)

def on_reset(b):
    game["level"] = 1
    game["score"] = 0
    game["total_attempts"] = 0
    setup_level()
    show_game("""
    <div style='text-align: center; padding: 20px;'>
        <h2 class='wave'>🌊 New Mission</h2>
        <p class='story'>"Somewhere in this quantum wire, an electron carries a secret."</p>
        <p class='story'>"It's not a dot. It's a wave of probability. Find it."</p>
        <p>Click any <b>Pos</b> button to probe the wave.</p>
    </div>
    """)

def on_lesson(b):
    show_game("""
    <div class='lesson'>
        <h3>📚 SCHRÖDINGER'S WAVE FUNCTION</h3>
        <p>Before quantum mechanics, we thought electrons were tiny balls orbiting the nucleus.</p>
        <p><b>Schrödinger changed everything in 1926.</b></p>
        <p>The electron is a <b>wave function ψ</b> — a mathematical cloud of probability.</p>
        <p><b>|ψ|²</b> tells you the probability of finding it at any location.</p>
        <p>Before measurement: the electron has no exact position. It's <b>everywhere and nowhere.</b></p>
        <p>When you measure: the wave <b>collapses</b> to a point. You FORCE it to pick a location.</p>
        <p><b>This game:</b> Each probe is a measurement. The wave collapses near where you probe. But never exactly. This is the <b>uncertainty principle</b> in action.</p>
        <p><b>In quantum networks (RAQT, BB84):</b> Eve's interception IS a measurement. It collapses the wave. Alice and Bob detect the collapse. Security is guaranteed by physics.</p>
    </div>
    <p style='color: #4fc3f7;'>⬇️ Click any Pos button to start probing!</p>
    """)

btn_reset.on_click(on_reset)
btn_lesson.on_click(on_lesson)

# ============================================================
# START GAME
# ============================================================
setup_level()
show_game("""
<div style='text-align: center; padding: 20px;'>
    <h2 class='wave'>🌊 RIPPLE: The Electron Heist</h2>
    <p class='story'>"The electron is not a marble. It's not in one place."</p>
    <p class='story'>"It's a WAVE — a cloud of probability spreading through the wire."</p>
    <p class='story'>"Your mission: find it before your probes run out."</p>
    <p class='story'>"But be careful — every probe disturbs the wave..."</p>
    <br>
    <p>Click <b>📚 What is the Wave?</b> to learn the physics, or start probing below.</p>
</div>
""")

# Display all buttons
button_rows = create_buttons()
display(widgets.HBox([btn_lesson, btn_reset]))
display(widgets.HBox(button_rows[:6]))
display(widgets.HBox(button_rows[6:]))
display(game_output)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.2 MB/s eta 0:00:00


Output()